# 12 --- HyDE Retrieval vs. Query Expansion (on the MedQA benchmark)

**New idea: HyDE (Hypothetical Document Embeddings).** Instead of searching with the raw question,
the model first writes a short *hypothetical German guideline-style answer*, then we search with **that**.
The hypothetical text carries the formal German clinical vocabulary the AWMF guidelines actually use,
so it usually retrieves better-matching chunks than the bare cross-lingual question.

We compare, head to head, on **your existing MedQA benchmark**:
- **Baseline** = translate + MeSH query expansion (your current method).
- **HyDE** = generate a hypothetical German passage, embed it, retrieve.

Reported **both on the full set and the answerable subset** (corpus-grounded GT), with the 4 RAGAS metrics.
Neon reconnect + judge max_tokens fixes are baked in. Single generator = Mistral.

## 1. Install

In [ ]:
!pip install -q ragas langchain langchain-openai langchain-huggingface psycopg2-binary pgvector langchain-postgres datasets nest_asyncio sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.0/213.0 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.4/252.4 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 

## 2. VertexAI import patch

In [ ]:
import sys, types
class DummyVertexAI: pass
class DummyChatVertexAI: pass
m = types.ModuleType("langchain_community.llms"); m.VertexAI = DummyVertexAI; sys.modules["langchain_community.llms"] = m
m = types.ModuleType("langchain_community.chat_models"); m.ChatVertexAI = DummyChatVertexAI; sys.modules["langchain_community.chat_models"] = m
m = types.ModuleType("langchain_community.chat_models.vertexai"); m.ChatVertexAI = DummyChatVertexAI; sys.modules["langchain_community.chat_models.vertexai"] = m
m = types.ModuleType("langchain_community.llms.vertexai"); m.VertexAI = DummyVertexAI; sys.modules["langchain_community.llms.vertexai"] = m

## 3. Setup: resilient DB engine, embedder, reranker, models, prompts

In [ ]:
import os, json, time, random
import pandas as pd, torch, nest_asyncio
from google.colab import userdata, drive
from sqlalchemy import create_engine
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGVector
from langchain_core.prompts import PromptTemplate
from sentence_transformers import CrossEncoder

nest_asyncio.apply()
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/'
df = pd.read_csv(DRIVE_PATH + 'AWMF_Golden_Dataset_200Q_Final.csv')
print("Loaded", len(df), "rows.")

NEON = userdata.get('NEON_DATABASE_URL')
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

# Resilient Neon engine (serverless drops idle SSL connections)
engine = create_engine(NEON, pool_pre_ping=True, pool_recycle=180, pool_size=2, max_overflow=2,
                       connect_args={"sslmode":"require","keepalives":1,"keepalives_idle":30,
                                     "keepalives_interval":10,"keepalives_count":5})

bge = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={'device':'cpu'})
vs = PGVector(embeddings=bge, collection_name="awmf_baseline_bge", connection=engine, use_jsonb=True)

K_RETRIEVE = 30
K_FINAL = 10           # the chosen k, consistent with notebooks 11 and 13
USE_RERANKER = True    # reranker ON -- MUST match the notebook-11 baseline run reused below
retriever = vs.as_retriever(search_kwargs={"k": K_RETRIEVE})

reranker = None
if USE_RERANKER:
    reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512,
                            device="cuda" if torch.cuda.is_available() else "cpu")
    print("reranker loaded")

def make_llm(model, max_tokens=1024):
    return ChatOpenAI(model=model, api_key=os.environ["OPENROUTER_API_KEY"],
                      base_url="https://openrouter.ai/api/v1", temperature=0,
                      max_tokens=max_tokens, max_retries=6, request_timeout=90)

mistral = make_llm("mistralai/mistral-large")

def safe_invoke(llm, prompt, max_tries=8, base=8):
    for a in range(max_tries):
        try:
            return llm.invoke(prompt).content.strip()
        except Exception as e:
            if a == max_tries-1: raise
            w = min(base*(2**a)+random.uniform(0,3), 120)
            print(f"  retry {a+1}: {str(e)[:70]} ... {w:.0f}s"); time.sleep(w)

# --- Prompts ---
# Baseline: translate + MeSH expansion (your current method)
expand_prompt = PromptTemplate(template="""You are a medical search term generator.
Translate the English question to German, then add 3-4 formal German clinical synonyms / related conditions / MeSH terms.
Output ONLY the German question + synonyms as one continuous search string. No labels, no bullets.

English Question:
{question}""", input_variables=["question"])

# HyDE: write a hypothetical German guideline passage, which we embed for search
hyde_prompt = PromptTemplate(template="""You are a German medical guideline expert.
Write a short, factual German passage (3-5 sentences) that would plausibly answer the clinical question below,
as if quoted from an official clinical guideline. Use formal German medical terminology.
Do NOT say you are unsure; write the most likely guideline-style content. Output ONLY the German passage.

Clinical question (English):
{question}""", input_variables=["question"])

qa_prompt = PromptTemplate(template="""You are an expert medical AI. Read the German clinical guidelines and answer the medical question in ENGLISH.
Use ONLY the provided German context. If the context does not contain the answer, say so plainly.

Context (German):
{context}

Question (English):
{question}

Answer (English):""", input_variables=["context","question"])

print("Setup complete.")

Mounted at /content/drive
Loaded 200 rows.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

reranker loaded
Setup complete.


## 4. Retrieval functions: baseline expansion vs. HyDE

In [ ]:
def _rerank_or_top(query, texts, k_final):
    if reranker is not None and texts:
        scores = reranker.predict([[query, t] for t in texts])
        texts = [t for _, t in sorted(zip(scores, texts), key=lambda x: x[0], reverse=True)]
    return texts[:k_final]

def retrieve_baseline(question, k_final=K_FINAL):
    gq = safe_invoke(mistral, expand_prompt.format(question=question))
    docs = retriever.invoke(gq)
    return _rerank_or_top(gq, [d.page_content for d in docs], k_final)

def retrieve_hyde(question, k_final=K_FINAL):
    passage = safe_invoke(mistral, hyde_prompt.format(question=question))   # the "think first" step
    docs = retriever.invoke(passage)                                        # search with the hypothetical answer
    return _rerank_or_top(passage, [d.page_content for d in docs], k_final)

# smoke test
_q = df.iloc[0]['English_Open_Question']
print("baseline top:", retrieve_baseline(_q)[0][:160], "...")
print("hyde     top:", retrieve_hyde(_q)[0][:160], "...")

baseline top: NVL Asthma 
Langfassung – Version 5.0 
 © NVL-Programm 2024      Seite 28 
Klinischer Hinweis Mögliche Diagnose 
Anfallartiger Husten Pertussis/postinfektiöse  ...
hyde     top: Myokardinfarkt oder Myokarditis 
Blutzucker, HbA1C: bei V. a. Diabetes mellitus 
Differentialblutbild: bei V. a. Anämie; 
Ferritin und Transferrinsättigung: Dif ...


## 5. Generate HyDE answers for all 200 (baseline reused from notebook 11)

Only **HyDE** is generated here, on **all 200 MedQA questions** (resumable). The baseline is **not** regenerated:
its answers are loaded from notebook 11's results file, which was produced with the same k, the same reranker
setting, and the same QA prompt over the same 200 questions -- so reusing it saves OpenRouter calls and keeps
a single canonical baseline. Both ground truths are stored per question, so the same answers are scored two
ways in the next section.

**Requirement:** notebook 11 must have finished its full 200-question run at `DEV=False` with the matching
`USE_RERANKER`, otherwise the baseline file below will be missing or incomplete.

In [ ]:
# Corpus-grounded GT from notebook 11 (defines which questions are answerable)
gt_df = pd.read_csv(DRIVE_PATH + "AWMF_CorpusGrounded_GroundTruth.csv")
gt_map = gt_df.set_index('English_Open_Question')['corpus_ground_truth'].to_dict()
answerable = {q for q, a in gt_map.items() if a != 'NOT_IN_CORPUS'}
work = df.reset_index(drop=True)   # ALL 200 questions
print(f"Generating HyDE on all {len(work)} MedQA questions  |  {len(answerable)} are corpus-answerable")

# Reuse notebook 11's baseline (same naming as notebook 11: tag depends on the reranker setting).
base_tag = ("rerank" if USE_RERANKER else "norerank") + "_full"
BASELINE_FILE = DRIVE_PATH + f"MISTRAL_{base_tag}_results.json"
assert os.path.exists(BASELINE_FILE), (
    f"Baseline not found: {BASELINE_FILE}. Run notebook 11 first (DEV=False, "
    f"USE_RERANKER={USE_RERANKER}) so its baseline can be reused here.")
_b = json.load(open(BASELINE_FILE))
print(f"Reusing notebook-11 baseline: {len(_b['question'])} answers from {BASELINE_FILE}")
if len(_b['question']) < len(df):
    print(f"WARNING: baseline has only {len(_b['question'])}/{len(df)} answers -- finish notebook 11 for a full comparison.")

def run_hyde():
    out_file = DRIVE_PATH + "HYDE_hyde_results.json"
    if os.path.exists(out_file):
        res = json.load(open(out_file)); done = set(res["question"])
        print(f"  resuming -- {len(done)} already done")
    else:
        res = {"question":[],"answer":[],"contexts":[],"medqa_ground_truth":[],"corpus_ground_truth":[]}; done = set()
    for i in range(len(work)):
        r = work.iloc[i]; q = r['English_Open_Question']
        if q in done: continue
        try:
            ctx = retrieve_hyde(q)
            ans = safe_invoke(mistral, qa_prompt.format(context="\n\n".join(ctx), question=q))
            res["question"].append(q); res["answer"].append(ans); res["contexts"].append(ctx)
            res["medqa_ground_truth"].append(r['English_Correct_Text'])
            res["corpus_ground_truth"].append(gt_map.get(q, "NOT_IN_CORPUS"))
            done.add(q)
            json.dump(res, open(out_file,"w"))
            if len(res["question"]) % 10 == 0: print(f"  [hyde] {len(res['question'])}/{len(work)}")
            time.sleep(2)
        except Exception as e:
            print("err", i, str(e)[:100]); time.sleep(8)
    print(f"hyde done -> {len(res['question'])}/{len(work)} -> {out_file}")
    return out_file

print("=== HyDE (baseline reused from notebook 11, not regenerated) ===")
run_hyde()

Generating HyDE on all 200 MedQA questions  |  49 are corpus-answerable
Reusing notebook-11 baseline: 200 answers from /content/drive/MyDrive/MISTRAL_rerank_full_results.json
=== HyDE (baseline reused from notebook 11, not regenerated) ===
  resuming -- 49 already done
  [hyde] 50/200
  [hyde] 60/200
  [hyde] 70/200
  [hyde] 80/200
  [hyde] 90/200
  [hyde] 100/200
  [hyde] 110/200
  [hyde] 120/200
  [hyde] 130/200
  [hyde] 140/200
  [hyde] 150/200
  [hyde] 160/200
  [hyde] 170/200
  [hyde] 180/200
  [hyde] 190/200
  [hyde] 200/200
hyde done -> 200/200 -> /content/drive/MyDrive/HYDE_hyde_results.json


'/content/drive/MyDrive/HYDE_hyde_results.json'

## 6. RAGAS evaluation --- baseline (from notebook 11) vs. HyDE, scored two ways

The baseline answers come from notebook 11 (no regeneration); only HyDE was generated here. The same answers
are scored under three views: the **full 200** against the MedQA ground truth, and the **answerable subset**
against the corpus-grounded ground truth (plus a MedQA parity check on that subset). The corpus metric exists
only where a corpus reference exists, so it is necessarily the answerable subset; the MedQA metric covers all
200. The judge runs with `max_tokens=4096` to avoid truncation.

In [ ]:
from datasets import Dataset, Features, Value, Sequence
from ragas import evaluate
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

judge = LangchainLLMWrapper(make_llm("openai/gpt-4o-mini", max_tokens=4096))   # <-- fixes truncation
remb = LangchainEmbeddingsWrapper(bge)
feat = Features({"question":Value("string"),"answer":Value("string"),
                 "contexts":Sequence(Value("string")),"ground_truth":Value("string")})

def score(path, gt_key, answerable_only=False):
    d = json.load(open(path))
    keep = list(range(len(d["question"])))
    # corpus GT only exists for answerable questions; drop NOT_IN_CORPUS rows when scoring against it.
    if gt_key == "corpus_ground_truth" or answerable_only:
        keep = [i for i in keep if d["corpus_ground_truth"][i] != "NOT_IN_CORPUS"]
    dd = {"question":[d["question"][i] for i in keep], "answer":[d["answer"][i] for i in keep],
          "contexts":[d["contexts"][i] for i in keep], "ground_truth":[d[gt_key][i] for i in keep]}
    ds = Dataset.from_dict(dd, features=feat)
    out = evaluate(ds, metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
                   llm=judge, embeddings=remb, run_config=RunConfig(timeout=300, max_workers=2, max_retries=5))
    means = out.to_pandas()[["context_precision","context_recall","faithfulness","answer_relevancy"]].mean().round(3)
    return dict(means), len(keep)

BASE = BASELINE_FILE                              # notebook-11 baseline
HYDE = DRIVE_PATH + "HYDE_hyde_results.json"      # generated above
def show(label, path, gt_key, answerable_only=False):
    m, n = score(path, gt_key, answerable_only)
    print(f"{label:9s} (n={n}):", m)

print("=== FULL 200 -- MedQA ground truth (the whole benchmark) ===")
show("BASELINE", BASE, "medqa_ground_truth"); show("HyDE", HYDE, "medqa_ground_truth")
print("\n=== ANSWERABLE SUBSET -- corpus-grounded ground truth (fair retrieval measure) ===")
show("BASELINE", BASE, "corpus_ground_truth"); show("HyDE", HYDE, "corpus_ground_truth")
print("\n=== ANSWERABLE SUBSET -- MedQA ground truth (parity check) ===")
show("BASELINE", BASE, "medqa_ground_truth", True); show("HyDE", HYDE, "medqa_ground_truth", True)

/tmp/ipykernel_2631/3556906884.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
/tmp/ipykernel_2631/3556906884.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
/tmp/ipykernel_2631/3556906884.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import context_precis

=== FULL 200 -- MedQA ground truth (the whole benchmark) ===


Evaluating:   0%|          | 0/800 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[22]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception raised in Job[782]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)


BASELINE  (n=200): {'context_precision': np.float64(0.181), 'context_recall': np.float64(0.225), 'faithfulness': np.float64(0.532), 'answer_relevancy': np.float64(0.254)}


...

HyDE      (n=49): {'context_precision': np.float64(0.675), 'context_recall': np.float64(0.816), 'faithfulness': np.float64(0.595), 'answer_relevancy': np.float64(0.522)}

=== ANSWERABLE SUBSET -- MedQA ground truth (parity check) ===


Evaluating:   0%|          | 0/196 [00:00<?, ?it/s]

BASELINE  (n=49): {'context_precision': np.float64(0.435), 'context_recall': np.float64(0.429), 'faithfulness': np.float64(0.637), 'answer_relevancy': np.float64(0.52)}


Evaluating:   0%|          | 0/196 [00:00<?, ?it/s]

HyDE      (n=49): {'context_precision': np.float64(0.511), 'context_recall': np.float64(0.449), 'faithfulness': np.float64(0.602), 'answer_relevancy': np.float64(0.511)}


### RAGAS Evaluation Comparison

| Metric             | Baseline (Full 200) | HyDE (Full 200) | Baseline (Answerable Subset - Corpus GT) | HyDE (Answerable Subset - Corpus GT) | Baseline (Answerable Subset - MedQA GT) | HyDE (Answerable Subset - MedQA GT) |
| :----------------- | :------------------ | :-------------- | :--------------------------------------- | :----------------------------------- | :-------------------------------------- | :---------------------------------- |
| **context_precision**| 0.181               | 0.205           | 0.685                                    | 0.675                                | 0.435                                   | 0.511                               |
| **context_recall**   | 0.225               | 0.273           | 0.819                                    | 0.816                                | 0.429                                   | 0.449                               |
| **faithfulness**   | 0.532               | 0.530           | 0.626                                    | 0.595                                | 0.637                                   | 0.602                               |
| **answer_relevancy** | 0.254               | 0.290           | 0.512                                    | 0.522                                | 0.520                                   | 0.511                               |